# AutoML

In [1]:
import sys
sys.path.append("../../")
sys.path.append("../../outboxml")

In [2]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import json

In [3]:
from outboxml.extractors import Extractor

python-dotenv could not parse statement starting at line 2


In [4]:
class DataExtractor(Extractor):
    def __init__(self):
        super().__init__()
    def extract_dataset(self):
        housing = fetch_california_housing(as_frame=True)
        return pd.concat([housing['data'], housing['target']],axis=1)

In [12]:
class LogsExtractor(Extractor):
    def extract_dataset(self) -> pd.DataFrame:
        housing = fetch_california_housing(as_frame=True)
        return pd.concat([housing['data'], housing['target']],axis=1)[:500]

In [5]:
from outboxml.automl_utils import build_default_all_models_config

In [6]:
model_params =  {'objective': 'gamma', 'wrapper': 'glm', 'name': 'price'}
features_params ={'encoding_num': 'WoE_num_cat'}

In [7]:
from outboxml.automl_manager import AutoMLManager

/Users/dima/miniforge3/envs/outboxml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
all_models_config_name = 'configs/house_pricing.json'

In [9]:
all_models_config = build_default_all_models_config(data=DataExtractor().extract_dataset(),
                                                    column_target='MedHouseVal', 
                                                    group_name='example',
                                                    project='house_pricing',
                                                    model_params = model_params,
                                                    features_params=features_params,
                                                   ) 
with open(all_models_config_name, 'w') as f:
    json.dump(dict(all_models_config.model_dump()), f)

2026-01-29 21:39:36.598 | INFO     | outboxml.core.config_builders:feature_params:170 - Prepare feature||MedInc
2026-01-29 21:39:36.600 | INFO     | outboxml.core.config_builders:feature_params:206 - {'type': 'numerical', 'name': 'MedInc', 'clip': {'min_value': 0.536, 'max_value': 15.0}, 'default': '_MEDIAN_', 'encoding': 'WoE_num_cat'}
2026-01-29 21:39:36.600 | DEBUG    | outboxml.core.config_builders:build:75 - Feature builder||MedInc
2026-01-29 21:39:36.600 | INFO     | outboxml.core.config_builders:feature_params:170 - Prepare feature||HouseAge
2026-01-29 21:39:36.602 | INFO     | outboxml.core.config_builders:feature_params:206 - {'type': 'numerical', 'name': 'HouseAge', 'clip': {'min_value': 2.0, 'max_value': 52.0}, 'default': '_MEDIAN_', 'encoding': 'WoE_num_cat'}
2026-01-29 21:39:36.602 | DEBUG    | outboxml.core.config_builders:build:75 - Feature builder||HouseAge
2026-01-29 21:39:36.603 | INFO     | outboxml.core.config_builders:feature_params:170 - Prepare feature||AveRooms


In [12]:
auto_ml = AutoMLManager(auto_ml_config='configs/automl-house_pricing.json',
                        models_config=all_models_config_name,
                        extractor=DataExtractor(),
                        retro=True,
                        hp_tune=False)

2026-01-29 21:50:12.609 | DEBUG    | outboxml.datasets_manager:_init_dsmanager:860 - Initializing DSManager
2026-01-29 21:50:12.611 | INFO     | outboxml.datasets_manager:__load_all_models_config:747 - All models config from path
2026-01-29 21:50:12.613 | WARNING  | outboxml.datasets_manager:__load_all_models_config:768 - price||File /Users/dima/PycharmProjects/outboxml/results/price_v1_subset.pickle already exists. Change version in config file to for new data prepare
2026-01-29 21:50:12.614 | INFO     | outboxml.datasets_manager:__load_all_models_config:775 - Config is loaded
2026-01-29 21:50:12.614 | INFO     | outboxml.datasets_manager:__load_prepare_datasets:805 - Load models prepare datasets
2026-01-29 21:50:12.615 | INFO     | outboxml.datasets_manager:_init_dsmanager:868 - Reading user extractor
2026-01-29 21:50:12.615 | DEBUG    | outboxml.datasets_manager:_init_dsmanager:876 - Initializing completed
2026-01-29 21:50:12.617 | INFO     | outboxml.automl_manager:__init_auto_ml:9

In [13]:
auto_ml.update_models(send_mail=False)

2026-01-29 21:50:15.609 | DEBUG    | outboxml.automl_manager:feature_selection:580 - Feature selection||Started
2026-01-29 21:50:15.621 | INFO     | outboxml.data_subsets:save_parquet:910 - ||Saving dataset to parquet
2026-01-29 21:50:15.633 | INFO     | outboxml.data_subsets:dataset:534 - Reading data from parquet
2026-01-29 21:50:15.638 | INFO     | outboxml.dataset_retro:features_for_reserch:103 - feature list to exclude from config ['general']
2026-01-29 21:50:15.638 | INFO     | outboxml.dataset_retro:features_for_reserch:112 - Retro||Features for research []
2026-01-29 21:50:15.639 | INFO     | outboxml.data_subsets:dataset:534 - Reading data from parquet
2026-01-29 21:50:15.643 | DEBUG    | outboxml.feature_selection:select_features:206 - Feature selection||Prepare of new_features for research
2026-01-29 21:50:15.644 | INFO     | outboxml.data_subsets:dataset:534 - Reading data from parquet
0it [00:00, ?it/s]
2026-01-29 21:50:15.660 | INFO     | outboxml.feature_selection:featur

Report saved to: /Users/dima/PycharmProjects/outboxml/results/automl_report.html


# Monitoring

In [9]:
from outboxml.monitoring_manager import MonitoringManager

In [13]:
monitoring = MonitoringManager(
    monitoring_config='configs/house_cost_monitoring_config.json',
    models_config=all_models_config_name,
    data_extractor=DataExtractor(),
    logs_extractor=LogsExtractor(),
)

2026-01-29 23:33:45.811 | DEBUG    | outboxml.datasets_manager:_init_dsmanager:860 - Initializing DSManager
2026-01-29 23:33:45.812 | INFO     | outboxml.datasets_manager:__load_all_models_config:747 - All models config from path
2026-01-29 23:33:45.814 | WARNING  | outboxml.datasets_manager:__load_all_models_config:768 - price||File /Users/dima/PycharmProjects/outboxml/results/price_v1_subset.pickle already exists. Change version in config file to for new data prepare
2026-01-29 23:33:45.814 | INFO     | outboxml.datasets_manager:__load_all_models_config:775 - Config is loaded
2026-01-29 23:33:45.815 | INFO     | outboxml.datasets_manager:__load_prepare_datasets:805 - Load models prepare datasets
2026-01-29 23:33:45.815 | INFO     | outboxml.datasets_manager:_init_dsmanager:868 - Reading user extractor
2026-01-29 23:33:45.815 | DEBUG    | outboxml.datasets_manager:_init_dsmanager:876 - Initializing completed
2026-01-29 23:33:45.816 | INFO     | outboxml.monitoring_manager:__init_monit

In [14]:
monitoring.review(
    send_mail=False,
    to_grafana=True
)

2026-01-29 23:33:46.810 | DEBUG    | outboxml.datasets_manager:_init_dsmanager:860 - Initializing DSManager
2026-01-29 23:33:46.810 | INFO     | outboxml.datasets_manager:__load_all_models_config:747 - All models config from path
2026-01-29 23:33:46.812 | WARNING  | outboxml.datasets_manager:__load_all_models_config:770 - price||File /Users/dima/PycharmProjects/outboxml/results/price_v1_subset.pickle already exists. Changing version in config file for A/B test
2026-01-29 23:33:46.813 | INFO     | outboxml.datasets_manager:__load_all_models_config:775 - Config is loaded
2026-01-29 23:33:46.814 | INFO     | outboxml.datasets_manager:__load_prepare_datasets:824 - User models prepare datasets
2026-01-29 23:33:46.814 | INFO     | outboxml.datasets_manager:_init_dsmanager:868 - Reading user extractor
2026-01-29 23:33:46.814 | DEBUG    | outboxml.datasets_manager:_init_dsmanager:876 - Initializing completed
2026-01-29 23:33:46.828 | INFO     | outboxml.data_subsets:save_parquet:910 - ||Saving

In [16]:
monitoring.result.reviews

{'datadrift': {'price':                    PSI          KL        JS
  HouseAge      8.661346  124.069586  9.136670
  Longitude     2.383307   -0.193114  0.235310
  AveRooms      0.148941    0.038127  0.014027
  AveBedrms     0.037709   -0.005427  0.004539
  Latitude      2.616692   -0.175899  0.236616
  AveOccup      0.210775    0.056181  0.022936
  MedInc        0.527015    0.191272  0.049546
  Population  179.610670    0.090122  0.035881}}

# ResultExport

In [ ]:
from outboxml.export_results import ResultExport

In [ ]:
ResultExport(auto_ml).plots(model_name='price', features=['MedInc'], )

In [ ]:
ResultExport(auto_ml).plots(model_name='price', features=['HouseAge'], )

In [ ]:
ResultExport(auto_ml).plots(model_name='price', features=['AveOccup'], )